# 问题三：含日内光伏预报更新的微网滚动购电模型

本 Notebook 给出一个可复现的 **日前计划 + 日内滚动修正（MPC）** 模型，并直接生成官方模板格式的 `result3.xlsx`。

核心回答是：

1. 0:00 根据日前负荷预测、0:00 光伏预报及历史预测残差，确定 144 个 10 分钟时段的计划购电量；
2. 6:00、12:00、18:00 使用最新光伏预报和已经观测到的当日负荷，重算当天剩余时段的调整购电量与储能计划；
3. 对比“不更新”和“滚动更新”两种策略的真实结算成本，判断是否值得引入日内预报；
4. 输出题目要求的表 1、表 2、表 3，并写入完整结果文件。

所有预测与调度均严格按时间顺序进行：第 (d) 天的决策只使用第 (d) 天决策时刻之前已经可获得的信息。

## 1. 符号、费用与关键假设

每个结算时段长度为 (Delta t=1/6) 小时。对时段 (t)：

- (p_t)：电价（元/kWh）；
- (L_t,G_t)：负荷与光伏功率；
- (q_t^0)：0:00 制定的计划购电量；
- (q_t^k)：更新时刻 (k\in\{6,12,18\}) 制定的调整购电量；
- (u_t^k=(q_t^k-q_t^0)^+)，(v_t^k=(q_t^0-q_t^k)^+)：上调量和下调量；
- (c_t,d_t,S_t)：储能充电量、放电量和时段末储电量；
- (e_t,w_t)：紧急购电量和无法利用的富余电量。

结算成本解释为：计划量先按正常电价支付；相对计划量的上调部分按 (1.5p_t) 支付，下调部分产生 (0.5p_t) 的违约费用；真实运行中的短缺按 (5p_t) 紧急购电。于是

\[
C=\sum_t p_tq_t^0+\sum_t p_t(1.5u_t+0.5v_t)+\sum_t5p_te_t.
\]

储能双向效率均取 90%，容量边界为 1200--10800 kWh，充放电功率不超过 5000 kW。为避免跨日末端价值被任意透支，每天令 (S_{144}=S_0=6000\) kWh。富余电量允许弃用，但其已发生的计划或调整费用仍计入总成本。

In [1]:
from __future__ import annotations

import copy
import shutil
import time
import warnings
from itertools import combinations
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from openpyxl import load_workbook
from scipy.optimize import linprog
from scipy.sparse import lil_matrix

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "font.size": 9})

In [2]:
ROOT = Path.cwd()
DATA_DIR = ROOT / "q1_data"
OUTPUT_DIR = ROOT / "q3_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def workbook_shape(path: Path) -> tuple[int, list[tuple[int, int]]]:
    '''Return worksheet count and worksheet dimensions without relying on non-English filenames.'''
    workbook = load_workbook(path, read_only=True, data_only=False)
    dimensions = [(sheet.max_row, sheet.max_column) for sheet in workbook.worksheets]
    workbook.close()
    return len(dimensions), dimensions


def find_source_workbook(sheet_count: int, dimensions: list[tuple[int, int]]) -> Path:
    '''Locate one attachment by its unique workbook structure.'''
    matches = []
    for candidate in DATA_DIR.glob("*.xlsx"):
        if candidate.name.startswith("~$"):
            continue
        if candidate.stem.lower().startswith("result"):
            continue
        count, shape = workbook_shape(candidate)
        if count == sheet_count and shape == dimensions:
            matches.append(candidate)
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one workbook with shape {dimensions}, found {matches}")
    return matches[0].resolve()


PRICE_FILE = find_source_workbook(1, [(145, 4)])
HISTORY_FILE = find_source_workbook(2, [(366, 145), (366, 145)])
PV_FORECAST_FILE = find_source_workbook(1, [(1461, 26)])
TEMPLATE_FILE = (DATA_DIR / "result3.xlsx").resolve()
RESULT_FILE = OUTPUT_DIR / "result3.xlsx"

print("Price workbook:", PRICE_FILE.name)
print("Historical load and PV workbook:", HISTORY_FILE.name)
print("PV forecast workbook:", PV_FORECAST_FILE.name)
print("Result template:", TEMPLATE_FILE)

Price workbook: 附件1.xlsx
Historical load and PV workbook: 附件2.xlsx
PV forecast workbook: 附件3.xlsx
Result template: C:\Users\lenovo\Downloads\CUMCM2026Problems\C题\q1_data\result3.xlsx


In [3]:
INTERVALS_PER_DAY = 144
INTERVAL_HOURS = 1.0 / 6.0
BATTERY_CAPACITY_KWH = 12_000.0
SOC_MIN_KWH = 1_200.0
SOC_MAX_KWH = 10_800.0
SOC_TARGET_KWH = 6_000.0
BATTERY_POWER_KW = 5_000.0
BATTERY_INTERVAL_LIMIT_KWH = BATTERY_POWER_KW * INTERVAL_HOURS
CHARGE_EFFICIENCY = 0.90
DISCHARGE_EFFICIENCY = 0.90
EMERGENCY_MULTIPLIER = 5.0

ISSUE_HOURS = (0, 6, 12, 18)
PLAN_RESIDUAL_QUANTILE = 0.80
ADJUSTMENT_RESIDUAL_QUANTILE = 0.70
RESIDUAL_WINDOW_DAYS = 30
MIN_FORECAST_DAY = 14
TIE_BREAK_COST = 1e-7

REPORT_DATES = ("2025-03-20", "2025-06-21", "2025-09-23", "2025-12-21")
REPORT_INTERVALS = (
    "10:00-10:10", "12:00-12:10", "14:00-14:10",
    "16:00-16:10", "18:00-18:10", "20:00-20:10",
)
FOUR_HOUR_LABELS = (
    "0:00-4:00", "4:00-8:00", "8:00-12:00",
    "12:00-16:00", "16:00-20:00", "20:00-24:00",
)

template_book = load_workbook(TEMPLATE_FILE, read_only=True, data_only=False)
PLAN_SHEET_NAME, ADJUSTMENT_SHEET_NAME, BATTERY_SHEET_NAME, EMERGENCY_SHEET_NAME = template_book.sheetnames
template_plan_sheet = template_book[PLAN_SHEET_NAME]
template_header = [template_plan_sheet.cell(1, column).value for column in range(1, 148)]
INTERVAL_LABELS = [str(value).strip() for value in template_header[1:145]]
REPORT_DAY_INDEX = pd.DatetimeIndex(
    [pd.Timestamp(template_plan_sheet.cell(row, 1).value).normalize() for row in range(2, 336)]
)
template_book.close()

LABEL_TO_INDEX = {label: index for index, label in enumerate(INTERVAL_LABELS)}
SLOT_END_HOURS = np.arange(1, INTERVALS_PER_DAY + 1, dtype=float) * INTERVAL_HOURS
ISSUE_STARTS = {
    hour: (0 if hour == 0 else int(np.flatnonzero(np.isclose(SLOT_END_HOURS, hour))[0]))
    for hour in ISSUE_HOURS
}
BLOCK_BOUNDS = [ISSUE_STARTS[0], ISSUE_STARTS[6], ISSUE_STARTS[12], ISSUE_STARTS[18], INTERVALS_PER_DAY]

print("Evaluation dates:", REPORT_DAY_INDEX[0].date(), "to", REPORT_DAY_INDEX[-1].date())
print("Decision block bounds:", BLOCK_BOUNDS)
print("First and last interval labels:", INTERVAL_LABELS[0], INTERVAL_LABELS[-1])

Evaluation dates: 2025-02-01 to 2025-12-31
Decision block bounds: [0, 35, 71, 107, 144]
First and last interval labels: 0:10-0:20 0:00-0:10+1


## 2. 数据读取与时间对齐

附件 2 的 144 个功率点对应模板中的 144 个 10 分钟结算列。附件 3 每天在 0:00、6:00、12:00、18:00 给出未来 1--24 小时整点光伏功率。对任一发布时间，使用“发布时刻已经观测到的光伏功率 + 未来 24 个整点预报”作为节点，线性插值到 10 分钟网格。

负荷没有给定外部预报，因此使用最近 4 个同星期日的曲线作加权平均；在 6:00、12:00、18:00 再用当日已经观测到的最近两小时负荷比例进行有界修正，并让修正影响随预测步长指数衰减。

In [4]:
price_frame = pd.read_excel(PRICE_FILE)
PRICE = pd.to_numeric(price_frame.iloc[:, 1], errors="raise").to_numpy(float)

history_book = pd.ExcelFile(HISTORY_FILE)
load_frame = history_book.parse(history_book.sheet_names[0])
pv_frame = history_book.parse(history_book.sheet_names[1])

DATES = pd.DatetimeIndex(pd.to_datetime(load_frame.iloc[:, 0])).normalize()
LOAD_KW = load_frame.iloc[:, 1:].to_numpy(float)
PV_KW = pv_frame.iloc[:, 1:].to_numpy(float)
LOAD_KWH = LOAD_KW * INTERVAL_HOURS
PV_KWH = PV_KW * INTERVAL_HOURS
NET_KWH = LOAD_KWH - PV_KWH
DAY_OF_WEEK = DATES.dayofweek.to_numpy()
DATE_TO_DAY = {date: index for index, date in enumerate(DATES)}
EVAL_DAYS = np.array([DATE_TO_DAY[date] for date in REPORT_DAY_INDEX], dtype=int)

forecast_frame = pd.read_excel(PV_FORECAST_FILE)
forecast_dates = pd.to_datetime(forecast_frame.iloc[:, 0].replace("", np.nan).ffill()).dt.normalize()
forecast_issue_hours = (
    forecast_frame.iloc[:, 1].astype(str).str.extract(r"(\d+):", expand=False).astype(int)
)
forecast_values = forecast_frame.iloc[:, 2:26].apply(pd.to_numeric, errors="raise").to_numpy(float)

PV_HOURLY_FORECAST = np.full((len(DATES), len(ISSUE_HOURS), 24), np.nan)
issue_to_position = {hour: position for position, hour in enumerate(ISSUE_HOURS)}
for row in range(len(forecast_frame)):
    day = DATE_TO_DAY[pd.Timestamp(forecast_dates.iloc[row])]
    issue_position = issue_to_position[int(forecast_issue_hours.iloc[row])]
    PV_HOURLY_FORECAST[day, issue_position] = forecast_values[row]

validation = pd.Series(
    {
        "historical days": len(DATES),
        "intervals per day": LOAD_KW.shape[1],
        "evaluation days": len(EVAL_DAYS),
        "missing load values": int(np.isnan(LOAD_KW).sum()),
        "missing realized PV values": int(np.isnan(PV_KW).sum()),
        "missing hourly forecast values": int(np.isnan(PV_HOURLY_FORECAST).sum()),
        "minimum tariff (yuan/kWh)": PRICE.min(),
        "maximum tariff (yuan/kWh)": PRICE.max(),
        "usable battery energy (kWh)": SOC_MAX_KWH - SOC_MIN_KWH,
        "interval battery limit (kWh)": BATTERY_INTERVAL_LIMIT_KWH,
    },
    name="value",
)
display(validation.to_frame())

assert PRICE.shape == (INTERVALS_PER_DAY,)
assert LOAD_KW.shape == PV_KW.shape == (365, INTERVALS_PER_DAY)
assert np.isfinite(PV_HOURLY_FORECAST).all()
assert np.array_equal(EVAL_DAYS, np.arange(31, 365))
assert (LOAD_KW >= 0).all() and (PV_KW >= 0).all() and (PRICE > 0).all()
print("All source-data checks passed.")

,value
historical days,365.000000
intervals per day,144.000000
evaluation days,334.000000
missing load values,0.000000
missing realized PV values,0.000000
missing hourly forecast values,0.000000
minimum tariff (yuan/kWh),0.371300
maximum tariff (yuan/kWh),1.395200
usable battery energy (kWh),9600.000000
interval battery limit (kWh),833.333333


All source-data checks passed.


In [5]:
def base_load_forecast(day: int) -> np.ndarray:
    '''Forecast a full daily load profile from information available before the day.'''
    same_weekday = np.flatnonzero(DAY_OF_WEEK[:day] == DAY_OF_WEEK[day])[-4:]
    if same_weekday.size == 0:
        history = np.arange(max(0, day - 7), day)
    else:
        history = same_weekday
    weights = np.arange(1, len(history) + 1, dtype=float)
    weights /= weights.sum()
    return np.average(LOAD_KW[history], axis=0, weights=weights)


def load_forecast_at_issue(day: int, issue_hour: int) -> np.ndarray:
    '''Return the load forecast over the remaining horizon at one issue time.'''
    start = ISSUE_STARTS[issue_hour]
    base = base_load_forecast(day)
    remaining = base[start:].copy()
    if issue_hour == 0:
        return remaining

    observed_end = start
    observed_start = max(0, observed_end - 12)
    observed_actual = LOAD_KW[day, observed_start:observed_end]
    observed_base = base[observed_start:observed_end]
    level_ratio = observed_actual.sum() / max(observed_base.sum(), 1e-9)
    level_ratio = float(np.clip(level_ratio, 0.80, 1.20))
    elapsed = SLOT_END_HOURS[start:] - issue_hour
    decay = np.exp(-elapsed / 6.0)
    return remaining * (1.0 + (level_ratio - 1.0) * decay)


def pv_forecast_at_issue(day: int, issue_hour: int) -> np.ndarray:
    '''Interpolate the 24 hourly PV forecasts onto the remaining 10-minute grid.'''
    start = ISSUE_STARTS[issue_hour]
    issue_position = issue_to_position[issue_hour]
    hourly = PV_HOURLY_FORECAST[day, issue_position]
    if issue_hour == 0:
        anchor = PV_KW[day - 1, -1] if day > 0 else 0.0
    else:
        anchor = PV_KW[day, start]
    node_hours = np.arange(0, 25, dtype=float)
    node_values = np.concatenate([[anchor], hourly])
    target_hours = SLOT_END_HOURS[start:] - issue_hour
    return np.clip(np.interp(target_hours, node_hours, node_values), 0.0, None)


FORECASTS_BY_ISSUE = {}
for issue_hour in ISSUE_HOURS:
    start = ISSUE_STARTS[issue_hour]
    horizon = INTERVALS_PER_DAY - start
    load_forecasts = np.full((len(DATES), horizon), np.nan)
    pv_forecasts = np.full((len(DATES), horizon), np.nan)
    for day in range(MIN_FORECAST_DAY, len(DATES)):
        load_forecasts[day] = load_forecast_at_issue(day, issue_hour)
        pv_forecasts[day] = pv_forecast_at_issue(day, issue_hour)
    FORECASTS_BY_ISSUE[issue_hour] = {
        "load_kw": load_forecasts,
        "pv_kw": pv_forecasts,
        "net_kwh": (load_forecasts - pv_forecasts) * INTERVAL_HOURS,
    }

print("Causal load and PV forecast arrays constructed for all four issue times.")

Causal load and PV forecast arrays constructed for all four issue times.


## 3. 预测误差的因果分位数修正

单纯使用点预测会低估高价紧急购电风险。令 (r_{d,k,t}=N_{d,t}-\hat N_{d,k,t}) 为历史净负荷预测残差。对第 (d) 天，仅使用过去 30 天同一发布时间的残差，取逐时段经验分位数并作局部中位数平滑：

\[
\widetilde N_{d,k,t}=\hat N_{d,k,t}+Q_{\alpha_k}\{r_{j,k,t}:j<d\}.
\]

0:00 计划采用 (alpha_0=0.80)。其来源是新闻商临界分位数：少买 1 kWh 后需要以 5 倍电价补购，相对正常购电的边际损失为 (4p_t)，多买的边际损失为 (p_t)，故 (alpha_0=4/(4+1)=0.8)。日内上调的边际价格为 (1.5p_t)，相应分位数为 ((5-1.5)/5=0.7)。

In [6]:
def corrected_net_forecast(day: int, issue_hour: int, quantile: float) -> np.ndarray:
    '''Add a causal rolling residual quantile to the raw net-load forecast.'''
    start = ISSUE_STARTS[issue_hour]
    raw = FORECASTS_BY_ISSUE[issue_hour]["net_kwh"][day]
    history_start = max(MIN_FORECAST_DAY, day - RESIDUAL_WINDOW_DAYS)
    history_days = np.arange(history_start, day)
    historical_actual = NET_KWH[history_days, start:]
    historical_raw = FORECASTS_BY_ISSUE[issue_hour]["net_kwh"][history_days]
    residuals = historical_actual - historical_raw
    buffer = np.quantile(residuals, quantile, axis=0)
    buffer = pd.Series(buffer).rolling(7, center=True, min_periods=1).median().to_numpy()
    return raw + buffer


forecast_quality_rows = []
for issue_hour in ISSUE_HOURS:
    start = ISSUE_STARTS[issue_hour]
    actual_load = LOAD_KW[EVAL_DAYS, start:]
    actual_pv = PV_KW[EVAL_DAYS, start:]
    predicted_load = FORECASTS_BY_ISSUE[issue_hour]["load_kw"][EVAL_DAYS]
    predicted_pv = FORECASTS_BY_ISSUE[issue_hour]["pv_kw"][EVAL_DAYS]
    forecast_quality_rows.append(
        {
            "issue time": f"{issue_hour:02d}:00",
            "remaining points": predicted_pv.shape[1],
            "load MAE (kW)": np.abs(predicted_load - actual_load).mean(),
            "PV MAE (kW)": np.abs(predicted_pv - actual_pv).mean(),
            "net-load MAE (kW)": np.abs((predicted_load - predicted_pv) - (actual_load - actual_pv)).mean(),
        }
    )

forecast_quality = pd.DataFrame(forecast_quality_rows).set_index("issue time")
display(forecast_quality.round(2))

block_comparison_rows = []
zero_forecast = FORECASTS_BY_ISSUE[0]["net_kwh"][EVAL_DAYS]
for block, issue_hour in enumerate(ISSUE_HOURS):
    start, end = BLOCK_BOUNDS[block], BLOCK_BOUNDS[block + 1]
    actual = NET_KWH[EVAL_DAYS, start:end]
    initial = zero_forecast[:, start:end]
    latest = FORECASTS_BY_ISSUE[issue_hour]["net_kwh"][EVAL_DAYS, : end - start]
    block_comparison_rows.append(
        {
            "execution block": f"{issue_hour:02d}:00 to {ISSUE_HOURS[block + 1]:02d}:00" if block < 3 else "18:00 to day end",
            "0:00 net MAE (kWh/interval)": np.abs(initial - actual).mean(),
            "latest net MAE (kWh/interval)": np.abs(latest - actual).mean(),
        }
    )

block_forecast_comparison = pd.DataFrame(block_comparison_rows).set_index("execution block")
block_forecast_comparison["MAE reduction (%)"] = 100.0 * (
    1.0
    - block_forecast_comparison["latest net MAE (kWh/interval)"]
    / block_forecast_comparison["0:00 net MAE (kWh/interval)"]
)
display(block_forecast_comparison.round(2))

,remaining points,load MAE (kW),PV MAE (kW),net-load MAE (kW)
issue time,,,,
00:00,144,173.89,197.49,310.08
06:00,109,158.89,157.06,253.88
12:00,73,172.24,74.22,205.00
18:00,37,151.54,9.74,155.84


,0:00 net MAE (kWh/interval),latest net MAE (kWh/interval),MAE reduction (%)
execution block,,,
00:00 to 06:00,28.35,28.35,0.00
06:00 to 12:00,62.60,37.80,39.62
12:00 to 18:00,82.26,36.12,56.09
18:00 to day end,33.37,25.97,22.16


## 4. 两阶段滚动线性规划

### 4.1 0:00 日前计划

用修正后的净负荷 (widetilde N_t) 求解：

\[
\min \sum_t p_tq_t^0,
\]

\[
q_t^0+d_t-c_t-w_t=\widetilde N_t,
\qquad
S_t=S_{t-1}+0.9c_t-d_t/0.9.
\]

### 4.2 日内调整

在 (k\in\{6,12,18\}) 时，优化从当前时刻到下一次预报发布时刻的执行块；已执行部分和当前储能状态固定，块末储能状态取日前计划在同一时刻的状态作为终端参考：

\[
q_t^k-q_t^0=u_t^k-v_t^k,
\]

\[
\min \sum_{t\in B_k}p_t(1.5u_t^k+0.5v_t^k),
\qquad S_{\operatorname{end}(B_k)}=S^0_{\operatorname{end}(B_k)}.
\]

两类问题均满足 (q,c,d,w\ge0)、(1200\le S_t\le10800)、(0\le c_t,d_t\le833.33) kWh/时段及日末储能约束。目标中加入极小的正则项，仅用于消除多重最优解，不改变费用口径。

In [7]:
def solve_initial_schedule(
    target_net_kwh: np.ndarray,
    price: np.ndarray,
    opening_soc: float,
    terminal_soc: float,
) -> dict[str, np.ndarray]:
    '''Solve the 00:00 purchase and battery schedule as a linear program.'''
    horizon = len(target_net_kwh)
    purchase = slice(0, horizon)
    charge = slice(horizon, 2 * horizon)
    discharge = slice(2 * horizon, 3 * horizon)
    spill = slice(3 * horizon, 4 * horizon)
    soc = slice(4 * horizon, 5 * horizon)
    variable_count = 5 * horizon

    objective = np.zeros(variable_count)
    objective[purchase] = price
    objective[charge] = TIE_BREAK_COST
    objective[discharge] = TIE_BREAK_COST
    objective[spill] = TIE_BREAK_COST

    equality = lil_matrix((2 * horizon + 1, variable_count))
    rhs = np.zeros(2 * horizon + 1)
    for interval in range(horizon):
        equality[interval, purchase.start + interval] = 1.0
        equality[interval, discharge.start + interval] = 1.0
        equality[interval, charge.start + interval] = -1.0
        equality[interval, spill.start + interval] = -1.0
        rhs[interval] = target_net_kwh[interval]

        row = horizon + interval
        equality[row, soc.start + interval] = 1.0
        equality[row, charge.start + interval] = -CHARGE_EFFICIENCY
        equality[row, discharge.start + interval] = 1.0 / DISCHARGE_EFFICIENCY
        if interval == 0:
            rhs[row] = opening_soc
        else:
            equality[row, soc.start + interval - 1] = -1.0

    equality[-1, soc.stop - 1] = 1.0
    rhs[-1] = terminal_soc

    bounds = (
        [(0.0, None)] * horizon
        + [(0.0, BATTERY_INTERVAL_LIMIT_KWH)] * horizon
        + [(0.0, BATTERY_INTERVAL_LIMIT_KWH)] * horizon
        + [(0.0, None)] * horizon
        + [(SOC_MIN_KWH, SOC_MAX_KWH)] * horizon
    )
    solution = linprog(
        objective,
        A_eq=equality.tocsr(),
        b_eq=rhs,
        bounds=bounds,
        method="highs-ds",
    )
    if not solution.success:
        raise RuntimeError(f"Initial schedule failed: {solution.message}")
    vector = solution.x
    return {
        "purchase": vector[purchase],
        "charge": vector[charge],
        "discharge": vector[discharge],
        "spill": vector[spill],
        "soc": vector[soc],
    }


def solve_adjusted_schedule(
    target_net_kwh: np.ndarray,
    baseline_purchase: np.ndarray,
    price: np.ndarray,
    opening_soc: float,
    terminal_soc: float,
) -> dict[str, np.ndarray]:
    '''Solve one rolling adjustment over the remaining daily horizon.'''
    horizon = len(target_net_kwh)
    purchase = slice(0, horizon)
    upward = slice(horizon, 2 * horizon)
    downward = slice(2 * horizon, 3 * horizon)
    charge = slice(3 * horizon, 4 * horizon)
    discharge = slice(4 * horizon, 5 * horizon)
    spill = slice(5 * horizon, 6 * horizon)
    soc = slice(6 * horizon, 7 * horizon)
    variable_count = 7 * horizon

    objective = np.zeros(variable_count)
    objective[upward] = 1.5 * price
    objective[downward] = 0.5 * price
    objective[charge] = TIE_BREAK_COST
    objective[discharge] = TIE_BREAK_COST
    objective[spill] = TIE_BREAK_COST

    equality = lil_matrix((3 * horizon + 1, variable_count))
    rhs = np.zeros(3 * horizon + 1)
    for interval in range(horizon):
        equality[interval, purchase.start + interval] = 1.0
        equality[interval, upward.start + interval] = -1.0
        equality[interval, downward.start + interval] = 1.0
        rhs[interval] = baseline_purchase[interval]

        balance_row = horizon + interval
        equality[balance_row, purchase.start + interval] = 1.0
        equality[balance_row, discharge.start + interval] = 1.0
        equality[balance_row, charge.start + interval] = -1.0
        equality[balance_row, spill.start + interval] = -1.0
        rhs[balance_row] = target_net_kwh[interval]

        storage_row = 2 * horizon + interval
        equality[storage_row, soc.start + interval] = 1.0
        equality[storage_row, charge.start + interval] = -CHARGE_EFFICIENCY
        equality[storage_row, discharge.start + interval] = 1.0 / DISCHARGE_EFFICIENCY
        if interval == 0:
            rhs[storage_row] = opening_soc
        else:
            equality[storage_row, soc.start + interval - 1] = -1.0

    equality[-1, soc.stop - 1] = 1.0
    rhs[-1] = terminal_soc

    bounds = (
        [(0.0, None)] * horizon
        + [(0.0, None)] * horizon
        + [(0.0, None)] * horizon
        + [(0.0, BATTERY_INTERVAL_LIMIT_KWH)] * horizon
        + [(0.0, BATTERY_INTERVAL_LIMIT_KWH)] * horizon
        + [(0.0, None)] * horizon
        + [(SOC_MIN_KWH, SOC_MAX_KWH)] * horizon
    )
    solution = linprog(
        objective,
        A_eq=equality.tocsr(),
        b_eq=rhs,
        bounds=bounds,
        method="highs-ds",
    )
    if not solution.success:
        raise RuntimeError(f"Adjusted schedule failed: {solution.message}")
    vector = solution.x
    return {
        "purchase": vector[purchase],
        "upward": vector[upward],
        "downward": vector[downward],
        "charge": vector[charge],
        "discharge": vector[discharge],
        "spill": vector[spill],
        "soc": vector[soc],
    }

## 5. 全年时序回测

对 2025-02-01 至 2025-12-31 逐日执行：

- **静态策略**：全天沿用 0:00 的计划购电量和储能计划；
- **滚动策略**：0:00--6:00 沿用日前计划，之后每 6 小时使用最新预报重算剩余时段，只执行到下一次更新时刻。

真实负荷与真实光伏仅在时段结束后用于计算紧急购电或富余电量。若实际净负荷高于计划供给，则差额为紧急购电；反之记为富余电量。这样可以在完全相同的真实轨迹上比较两种策略。

In [8]:
def storage_trajectory(charge: np.ndarray, discharge: np.ndarray, opening_soc: float) -> np.ndarray:
    '''Reconstruct the interval-end storage trajectory.'''
    changes = CHARGE_EFFICIENCY * charge - discharge / DISCHARGE_EFFICIENCY
    return opening_soc + np.cumsum(changes)


def settle_schedule(
    plan_purchase: np.ndarray,
    final_purchase: np.ndarray,
    charge: np.ndarray,
    discharge: np.ndarray,
    actual_net_kwh: np.ndarray,
    price: np.ndarray,
) -> dict[str, np.ndarray | float]:
    '''Settle one realized day under the problem's three-part cost rule.'''
    shortage = actual_net_kwh - (final_purchase + discharge - charge)
    emergency = np.maximum(shortage, 0.0)
    spill = np.maximum(-shortage, 0.0)
    upward = np.maximum(final_purchase - plan_purchase, 0.0)
    downward = np.maximum(plan_purchase - final_purchase, 0.0)
    plan_cost = float(price @ plan_purchase)
    adjustment_cost = float(price @ (1.5 * upward + 0.5 * downward))
    emergency_cost = float(EMERGENCY_MULTIPLIER * price @ emergency)
    return {
        "emergency": emergency,
        "spill": spill,
        "upward": upward,
        "downward": downward,
        "plan_cost": plan_cost,
        "adjustment_cost": adjustment_cost,
        "emergency_cost": emergency_cost,
        "total_cost": plan_cost + adjustment_cost + emergency_cost,
    }


def allocate_result_arrays() -> dict[str, np.ndarray]:
    day_count = len(EVAL_DAYS)
    return {
        "plan": np.zeros((day_count, INTERVALS_PER_DAY)),
        "adjusted": np.zeros((day_count, INTERVALS_PER_DAY)),
        "charge": np.zeros((day_count, INTERVALS_PER_DAY)),
        "discharge": np.zeros((day_count, INTERVALS_PER_DAY)),
        "soc": np.zeros((day_count, INTERVALS_PER_DAY)),
        "emergency": np.zeros((day_count, INTERVALS_PER_DAY)),
        "spill": np.zeros((day_count, INTERVALS_PER_DAY)),
        "upward": np.zeros((day_count, INTERVALS_PER_DAY)),
        "downward": np.zeros((day_count, INTERVALS_PER_DAY)),
        "plan_cost": np.zeros(day_count),
        "adjustment_cost": np.zeros(day_count),
        "emergency_cost": np.zeros(day_count),
        "total_cost": np.zeros(day_count),
        "soc_open": np.full(day_count, SOC_TARGET_KWH),
        "soc_close": np.zeros(day_count),
    }


def store_settlement(result: dict[str, np.ndarray], row: int, settlement: dict) -> None:
    for key in ("emergency", "spill", "upward", "downward"):
        result[key][row] = settlement[key]
    for key in ("plan_cost", "adjustment_cost", "emergency_cost", "total_cost"):
        result[key][row] = settlement[key]

In [9]:
def solve_one_day(day: int) -> tuple[dict, dict]:
    '''Solve both benchmark policies for one independent daily subproblem.'''
    plan_target = corrected_net_forecast(day, 0, PLAN_RESIDUAL_QUANTILE)
    plan_solution = solve_initial_schedule(
        plan_target,
        PRICE,
        SOC_TARGET_KWH,
        SOC_TARGET_KWH,
    )
    plan_purchase = plan_solution["purchase"]
    static_settlement = settle_schedule(
        plan_purchase,
        plan_purchase,
        plan_solution["charge"],
        plan_solution["discharge"],
        NET_KWH[day],
        PRICE,
    )
    static_day = {
        "plan": plan_purchase,
        "adjusted": plan_purchase,
        "charge": plan_solution["charge"],
        "discharge": plan_solution["discharge"],
        "soc": plan_solution["soc"],
        "soc_close": float(plan_solution["soc"][-1]),
        **static_settlement,
    }

    final_purchase = plan_purchase.copy()
    final_charge = np.zeros(INTERVALS_PER_DAY)
    final_discharge = np.zeros(INTERVALS_PER_DAY)
    current_soc = SOC_TARGET_KWH

    for block, issue_hour in enumerate(ISSUE_HOURS):
        start, end = BLOCK_BOUNDS[block], BLOCK_BOUNDS[block + 1]
        block_length = end - start
        if issue_hour == 0:
            stage_solution = plan_solution
        else:
            adjusted_target = corrected_net_forecast(
                day,
                issue_hour,
                ADJUSTMENT_RESIDUAL_QUANTILE,
            )[:block_length]
            stage_solution = solve_adjusted_schedule(
                adjusted_target,
                plan_purchase[start:end],
                PRICE[start:end],
                current_soc,
                float(plan_solution["soc"][end - 1]),
            )
        final_purchase[start:end] = stage_solution["purchase"][:block_length]
        final_charge[start:end] = stage_solution["charge"][:block_length]
        final_discharge[start:end] = stage_solution["discharge"][:block_length]
        current_soc = float(stage_solution["soc"][block_length - 1])

    final_soc = storage_trajectory(final_charge, final_discharge, SOC_TARGET_KWH)
    rolling_settlement = settle_schedule(
        plan_purchase,
        final_purchase,
        final_charge,
        final_discharge,
        NET_KWH[day],
        PRICE,
    )
    rolling_day = {
        "plan": plan_purchase,
        "adjusted": final_purchase,
        "charge": final_charge,
        "discharge": final_discharge,
        "soc": final_soc,
        "soc_close": float(final_soc[-1]),
        **rolling_settlement,
    }
    return static_day, rolling_day


start_time = time.perf_counter()
daily_results = [solve_one_day(int(day)) for day in EVAL_DAYS]

STATIC_POLICY = allocate_result_arrays()
ROLLING_POLICY = allocate_result_arrays()
matrix_keys = ("plan", "adjusted", "charge", "discharge", "soc", "emergency", "spill", "upward", "downward")
scalar_keys = ("plan_cost", "adjustment_cost", "emergency_cost", "total_cost", "soc_close")
for row, (static_day, rolling_day) in enumerate(daily_results):
    for key in matrix_keys:
        STATIC_POLICY[key][row] = static_day[key]
        ROLLING_POLICY[key][row] = rolling_day[key]
    for key in scalar_keys:
        STATIC_POLICY[key][row] = static_day[key]
        ROLLING_POLICY[key][row] = rolling_day[key]

elapsed = time.perf_counter() - start_time
print(f"Full chronological backtest completed in {elapsed:.1f} seconds.")

Full chronological backtest completed in 7.3 seconds.


In [10]:
def policy_summary(result: dict[str, np.ndarray]) -> pd.Series:
    return pd.Series(
        {
            "planned energy (kWh)": result["plan"].sum(),
            "final purchased energy (kWh)": result["adjusted"].sum(),
            "upward adjustment (kWh)": result["upward"].sum(),
            "downward adjustment (kWh)": result["downward"].sum(),
            "emergency energy (kWh)": result["emergency"].sum(),
            "days with emergency purchase": int((result["emergency"].sum(axis=1) > 1e-6).sum()),
            "unused surplus (kWh)": result["spill"].sum(),
            "planned purchase cost (yuan)": result["plan_cost"].sum(),
            "adjustment cost (yuan)": result["adjustment_cost"].sum(),
            "emergency cost (yuan)": result["emergency_cost"].sum(),
            "total cost (yuan)": result["total_cost"].sum(),
        }
    )


def combine_update_subset(update_hours: tuple[int, ...]) -> dict[str, np.ndarray]:
    '''Combine independently feasible six-hour update blocks into one policy.'''
    result = allocate_result_arrays()
    result["plan"] = STATIC_POLICY["plan"].copy()
    result["adjusted"] = STATIC_POLICY["adjusted"].copy()
    result["charge"] = STATIC_POLICY["charge"].copy()
    result["discharge"] = STATIC_POLICY["discharge"].copy()

    for issue_hour in update_hours:
        block = ISSUE_HOURS.index(issue_hour)
        start, end = BLOCK_BOUNDS[block], BLOCK_BOUNDS[block + 1]
        for key in ("adjusted", "charge", "discharge"):
            result[key][:, start:end] = ROLLING_POLICY[key][:, start:end]

    result["soc"] = np.vstack(
        [storage_trajectory(result["charge"][row], result["discharge"][row], SOC_TARGET_KWH)
         for row in range(len(EVAL_DAYS))]
    )
    result["soc_close"] = result["soc"][:, -1]
    shortage = NET_KWH[EVAL_DAYS] - (
        result["adjusted"] + result["discharge"] - result["charge"]
    )
    result["emergency"] = np.maximum(shortage, 0.0)
    result["spill"] = np.maximum(-shortage, 0.0)
    result["upward"] = np.maximum(result["adjusted"] - result["plan"], 0.0)
    result["downward"] = np.maximum(result["plan"] - result["adjusted"], 0.0)
    result["plan_cost"] = STATIC_POLICY["plan_cost"].copy()
    result["adjustment_cost"] = (
        PRICE[None, :] * (1.5 * result["upward"] + 0.5 * result["downward"])
    ).sum(axis=1)
    result["emergency_cost"] = (
        EMERGENCY_MULTIPLIER * PRICE[None, :] * result["emergency"]
    ).sum(axis=1)
    result["total_cost"] = (
        result["plan_cost"] + result["adjustment_cost"] + result["emergency_cost"]
    )
    return result


update_subsets = [
    subset
    for subset_size in range(4)
    for subset in combinations(ISSUE_HOURS[1:], subset_size)
]
SUBSET_POLICIES = {subset: combine_update_subset(subset) for subset in update_subsets}


def subset_label(subset: tuple[int, ...]) -> str:
    if not subset:
        return "00:00 only"
    return " + ".join(f"{hour:02d}:00" for hour in subset)


comparison = pd.DataFrame(
    {subset_label(subset): policy_summary(policy) for subset, policy in SUBSET_POLICIES.items()}
).T
static_total = comparison.loc["00:00 only", "total cost (yuan)"]
comparison["saving vs static (yuan)"] = static_total - comparison["total cost (yuan)"]
comparison["saving vs static (%)"] = 100.0 * comparison["saving vs static (yuan)"] / static_total
comparison = comparison.sort_values("total cost (yuan)")
display(comparison.round(2))

BEST_UPDATE_SUBSET = min(
    SUBSET_POLICIES,
    key=lambda subset: float(SUBSET_POLICIES[subset]["total_cost"].sum()),
)
BEST_UPDATE_LABEL = subset_label(BEST_UPDATE_SUBSET)
FINAL_POLICY = SUBSET_POLICIES[BEST_UPDATE_SUBSET]
print("Lowest-cost update set:", BEST_UPDATE_LABEL)

monthly = pd.DataFrame(
    {
        "00:00 plan only": STATIC_POLICY["total_cost"],
        f"selected updates ({BEST_UPDATE_LABEL})": FINAL_POLICY["total_cost"],
    },
    index=REPORT_DAY_INDEX,
).resample("ME").sum()
monthly.index = monthly.index.strftime("%Y-%m")
display(monthly.round(2))

axis = monthly.plot(figsize=(11, 3.8), marker="o")
axis.set_title("Monthly realized electricity cost")
axis.set_ylabel("yuan")
axis.set_xlabel("month")
plt.xticks(rotation=45)
plt.tight_layout()
display(axis.figure)
plt.close(axis.figure)

,planned energy (kWh),final purchased energy (kWh),upward adjustment (kWh),downward adjustment (kWh),emergency energy (kWh),days with emergency purchase,unused surplus (kWh),planned purchase cost (yuan),adjustment cost (yuan),emergency cost (yuan),total cost (yuan),saving vs static (yuan),saving vs static (%)
06:00 + 12:00,22166054.83,22410235.24,244180.41,0.0,279286.68,334.0,3487250.65,13552570.33,237968.19,1103356.75,14893895.27,42085.49,0.28
12:00,22166054.83,22292463.35,126408.52,0.0,312031.33,332.0,3355635.64,13552570.33,127099.05,1226039.95,14905709.33,30271.42,0.20
06:00,22166054.83,22283826.73,117771.89,0.0,327023.63,334.0,3385963.24,13552570.33,110869.14,1260727.22,14924166.69,11814.06,0.08
06:00 + 12:00 + 18:00,22166054.83,22425714.58,259659.75,0.0,281306.50,334.0,3519864.97,13552570.33,264304.02,1110941.21,14927815.56,8165.19,0.05
00:00 only,22166054.83,22166054.83,0.00,0.0,359768.28,329.0,3254348.23,13552570.33,0.00,1383410.42,14935980.76,0.00,0.00
12:00 + 18:00,22166054.83,22307942.69,141887.85,0.0,314051.15,333.0,3388249.96,13552570.33,153434.88,1233624.41,14939629.62,-3648.87,-0.02
06:00 + 18:00,22166054.83,22299306.06,133251.23,0.0,329043.45,334.0,3418577.57,13552570.33,137204.98,1268311.68,14958086.99,-22106.23,-0.15
18:00,22166054.83,22181534.17,15479.34,0.0,361788.10,332.0,3286962.56,13552570.33,26335.83,1390994.89,14969901.05,-33920.29,-0.23


Lowest-cost update set: 06:00 + 12:00


,00:00 plan only,selected updates (06:00 + 12:00)
2025-02,1303306.20,1325365.50
2025-03,1237929.36,1227401.50
2025-04,1070909.91,1038575.60
2025-05,1098793.91,1091734.64
2025-06,1452963.58,1486169.51
2025-07,1690600.03,1673192.67
2025-08,1415854.23,1418404.28
2025-09,1285215.67,1276128.94
2025-10,1252550.51,1242980.03
2025-11,1349414.38,1348661.38


<Figure size 1210x418 with 1 Axes>

In [11]:
def feasibility_report(result: dict[str, np.ndarray]) -> pd.Series:
    previous_soc = np.column_stack([result["soc_open"], result["soc"][:, :-1]])
    storage_error = result["soc"] - (
        previous_soc
        + CHARGE_EFFICIENCY * result["charge"]
        - result["discharge"] / DISCHARGE_EFFICIENCY
    )
    realized_balance = (
        result["adjusted"]
        + result["emergency"]
        + PV_KWH[EVAL_DAYS]
        + result["discharge"]
        - LOAD_KWH[EVAL_DAYS]
        - result["charge"]
        - result["spill"]
    )
    return pd.Series(
        {
            "maximum absolute energy-balance error (kWh)": np.abs(realized_balance).max(),
            "maximum absolute storage-transition error (kWh)": np.abs(storage_error).max(),
            "minimum SOC (kWh)": result["soc"].min(),
            "maximum SOC (kWh)": result["soc"].max(),
            "maximum charge power (kW)": result["charge"].max() / INTERVAL_HOURS,
            "maximum discharge power (kW)": result["discharge"].max() / INTERVAL_HOURS,
            "maximum absolute day-end SOC error (kWh)": np.abs(result["soc_close"] - SOC_TARGET_KWH).max(),
            "negative final-purchase entries": int((result["adjusted"] < -1e-8).sum()),
            "simultaneous charge-discharge intervals": int(
                ((result["charge"] > 1e-6) & (result["discharge"] > 1e-6)).sum()
            ),
        },
        name="value",
    )


checks = feasibility_report(FINAL_POLICY)
display(checks.to_frame())

assert checks["maximum absolute energy-balance error (kWh)"] < 1e-5
assert checks["maximum absolute storage-transition error (kWh)"] < 1e-5
assert checks["minimum SOC (kWh)"] >= SOC_MIN_KWH - 1e-5
assert checks["maximum SOC (kWh)"] <= SOC_MAX_KWH + 1e-5
assert checks["maximum charge power (kW)"] <= BATTERY_POWER_KW + 1e-5
assert checks["maximum discharge power (kW)"] <= BATTERY_POWER_KW + 1e-5
assert checks["maximum absolute day-end SOC error (kWh)"] < 1e-5
assert checks["negative final-purchase entries"] == 0
assert checks["simultaneous charge-discharge intervals"] == 0
print("All physical and accounting checks passed.")

,value
maximum absolute energy-balance error (kWh),4.547474e-13
maximum absolute storage-transition error (kWh),1.818989e-12
minimum SOC (kWh),1.200000e+03
maximum SOC (kWh),1.080000e+04
maximum charge power (kW),5.000000e+03
maximum discharge power (kW),5.000000e+03
maximum absolute day-end SOC error (kWh),1.091394e-11
negative final-purchase entries,0.000000e+00
simultaneous charge-discharge intervals,0.000000e+00


All physical and accounting checks passed.


## 6. 题目指定日期的表 1、表 2 和表 3

下列三组输出与题面格式对应：

- 表 1：指定 10 分钟时段的计划购电量，以及全日计划量、计划费用、调整费用、紧急购电量和总费用；
- 表 2：每 4 小时充电量、放电量，以及 0:00/24:00 储电量；
- 表 3：最终发生的连续紧急购电时段及其合计购电量。

In [12]:
REPORT_ROWS = {
    date: int(np.flatnonzero(REPORT_DAY_INDEX == pd.Timestamp(date))[0])
    for date in REPORT_DATES
}


def block_totals(values: np.ndarray) -> np.ndarray:
    return values.reshape(6, 24).sum(axis=1)


def split_interval_label(label: str) -> tuple[str, str]:
    start, end = str(label).split("-", 1)
    return start.strip(), end.strip()


INTERVAL_START, INTERVAL_END = zip(*(split_interval_label(label) for label in INTERVAL_LABELS))


def merge_emergency_windows(values: np.ndarray, tolerance: float = 1e-6) -> list[tuple[str, float]]:
    '''Merge consecutive positive emergency intervals into reporting windows.'''
    active = np.flatnonzero(values > tolerance)
    if active.size == 0:
        return []
    windows = []
    start = int(active[0])
    previous = int(active[0])
    for interval in active[1:]:
        interval = int(interval)
        if interval != previous + 1:
            label = f"{INTERVAL_START[start]}-{INTERVAL_END[previous]}"
            windows.append((label, float(values[start : previous + 1].sum())))
            start = interval
        previous = interval
    label = f"{INTERVAL_START[start]}-{INTERVAL_END[previous]}"
    windows.append((label, float(values[start : previous + 1].sum())))
    return windows


table_1 = pd.DataFrame(index=list(REPORT_INTERVALS) + [
    "daily planned purchase (kWh)",
    "daily planned cost (yuan)",
    "daily final purchase (kWh)",
    "daily adjustment cost (yuan)",
    "daily emergency purchase (kWh)",
    "daily total cost (yuan)",
])
table_2_charge = pd.DataFrame(index=FOUR_HOUR_LABELS)
table_2_discharge = pd.DataFrame(index=FOUR_HOUR_LABELS)
table_2_soc = pd.DataFrame(index=["SOC at 0:00 (kWh)", "SOC at 24:00 (kWh)"])
table_3_rows = []

for date, row in REPORT_ROWS.items():
    plan = FINAL_POLICY["plan"][row]
    table_1[date] = [plan[LABEL_TO_INDEX[label]] for label in REPORT_INTERVALS] + [
        plan.sum(),
        FINAL_POLICY["plan_cost"][row],
        FINAL_POLICY["adjusted"][row].sum(),
        FINAL_POLICY["adjustment_cost"][row],
        FINAL_POLICY["emergency"][row].sum(),
        FINAL_POLICY["total_cost"][row],
    ]
    table_2_charge[date] = block_totals(FINAL_POLICY["charge"][row])
    table_2_discharge[date] = block_totals(FINAL_POLICY["discharge"][row])
    table_2_soc[date] = [FINAL_POLICY["soc_open"][row], FINAL_POLICY["soc_close"][row]]
    windows = merge_emergency_windows(FINAL_POLICY["emergency"][row])
    if not windows:
        table_3_rows.append({"date": date, "window": "None", "purchase (kWh)": 0.0})
    else:
        for window, energy in windows:
            table_3_rows.append({"date": date, "window": window, "purchase (kWh)": energy})

table_3 = pd.DataFrame(table_3_rows)

print("Table 1 - purchase results on the four requested dates")
display(table_1.round(2))
print("Table 2a - four-hour charge energy")
display(table_2_charge.round(2))
print("Table 2b - four-hour discharge energy")
display(table_2_discharge.round(2))
print("Table 2c - boundary storage energy")
display(table_2_soc.round(2))
print("Table 3 - emergency-purchase windows")
display(table_3.round(2))

Table 1 - purchase results on the four requested dates


,2025-03-20,2025-06-21,2025-09-23,2025-12-21
10:00-10:10,0.00,0.00,0.00,0.00
12:00-12:10,596.86,0.00,421.30,1050.29
14:00-14:10,0.00,0.00,0.00,0.00
16:00-16:10,635.60,176.93,501.17,848.55
18:00-18:10,745.13,490.34,806.29,744.53
20:00-20:10,0.00,0.00,0.00,0.00
daily planned purchase (kWh),71875.21,38521.38,71384.51,100295.33
daily planned cost (yuan),43888.50,22432.33,44199.32,64401.19
daily final purchase (kWh),72562.95,38780.35,71384.51,100295.33
daily adjustment cost (yuan),786.79,162.83,0.00,0.00


Table 2a - four-hour charge energy


,2025-03-20,2025-06-21,2025-09-23,2025-12-21
0:00-4:00,4500.00,0.00,4500.00,4500.00
4:00-8:00,833.33,0.00,833.33,1032.24
8:00-12:00,5638.01,1697.29,4407.35,5163.85
12:00-16:00,4376.69,8293.15,5658.26,7946.37
16:00-20:00,574.11,262.64,273.30,98.84
20:00-24:00,5333.33,5333.33,5333.33,5333.33


Table 2b - four-hour discharge energy


,2025-03-20,2025-06-21,2025-09-23,2025-12-21
0:00-4:00,0.00,0.00,0.00,0.00
4:00-8:00,5956.54,3982.07,5984.01,1446.00
8:00-12:00,2316.76,0.00,2107.01,6812.84
12:00-16:00,303.64,2.68,283.49,2601.62
16:00-20:00,5676.53,5906.21,5661.72,5631.98
20:00-24:00,2963.47,2734.03,2978.28,3008.02


Table 2c - boundary storage energy


,2025-03-20,2025-06-21,2025-09-23,2025-12-21
SOC at 0:00 (kWh),6000.0,6000.0,6000.0,6000.0
SOC at 24:00 (kWh),6000.0,6000.0,6000.0,6000.0


Table 3 - emergency-purchase windows


,date,window,purchase (kWh)
0,2025-03-20,0:30-0:40,27.76
1,2025-03-20,1:10-1:50,59.30
2,2025-03-20,3:10-3:40,69.30
3,2025-03-20,4:00-4:10,1.20
4,2025-03-20,5:00-5:30,13.02
5,2025-03-20,8:10-9:40,381.74
6,2025-03-20,9:50-12:00,639.41
7,2025-03-20,13:10-14:40,200.99
8,2025-03-20,15:50-16:30,74.66
9,2025-03-20,18:50-19:20,71.81


In [13]:
sample_date = "2025-06-21"
sample_row = REPORT_ROWS[sample_date]
hours = SLOT_END_HOURS

figure, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(hours, LOAD_KWH[EVAL_DAYS[sample_row]], label="realized load", linewidth=1.2)
axes[0].plot(hours, PV_KWH[EVAL_DAYS[sample_row]], label="realized PV", linewidth=1.2)
axes[0].plot(hours, FINAL_POLICY["plan"][sample_row], label="00:00 plan", linewidth=1.0)
axes[0].plot(hours, FINAL_POLICY["adjusted"][sample_row], label="final purchase", linewidth=1.0)
axes[0].bar(hours, FINAL_POLICY["emergency"][sample_row], width=0.14, label="emergency", color="tab:red")
axes[0].set_ylabel("kWh per interval")
axes[0].set_title(f"Rolling schedule on {sample_date}")
axes[0].legend(ncol=5, fontsize=7)

axes[1].bar(hours, FINAL_POLICY["charge"][sample_row], width=0.14, label="charge", color="tab:green")
axes[1].bar(hours, -FINAL_POLICY["discharge"][sample_row], width=0.14, label="discharge", color="tab:orange")
axes[1].set_ylabel("kWh per interval")
axes[1].legend(fontsize=7)

axes[2].plot(hours, FINAL_POLICY["soc"][sample_row], color="tab:purple", label="stored energy")
axes[2].axhline(SOC_MIN_KWH, color="tab:red", linestyle="--", linewidth=0.8)
axes[2].axhline(SOC_MAX_KWH, color="tab:red", linestyle="--", linewidth=0.8, label="SOC limits")
axes[2].set_xlabel("hour")
axes[2].set_ylabel("kWh")
axes[2].legend(fontsize=7)

for update_hour in ISSUE_HOURS[1:]:
    for axis in axes:
        axis.axvline(update_hour, color="grey", linestyle=":", linewidth=0.8)

figure.tight_layout()
display(figure)
plt.close(figure)

<Figure size 1320x880 with 3 Axes>

## 7. 写入 `result3.xlsx`

结果文件保持附件 5 模板的四个工作表和原有列顺序：

1. “计划购电量”：0:00 确定的 144 个计划量；末两列为全天计划量和计划购电费；
2. “调整购电量”：最终执行的 144 个购电量；末两列为全天最终购电量和调整相关费用；
3. “充放电量”：全年每天 6 个四小时时段的最终充放电量，以及 0:00/24:00 储电量；
4. “紧急购电量”：全年每天最终连续紧急购电时段及电量；没有紧急购电的日期写入一行 `None, 0`。

In [14]:
def copy_cell_style(source, destination) -> None:
    if source.has_style:
        destination._style = copy.copy(source._style)
    if source.number_format:
        destination.number_format = source.number_format
    destination.alignment = copy.copy(source.alignment)
    destination.protection = copy.copy(source.protection)


shutil.copy2(TEMPLATE_FILE, RESULT_FILE)
result_book = load_workbook(RESULT_FILE)
plan_sheet = result_book[PLAN_SHEET_NAME]
adjustment_sheet = result_book[ADJUSTMENT_SHEET_NAME]
battery_sheet = result_book[BATTERY_SHEET_NAME]
emergency_sheet = result_book[EMERGENCY_SHEET_NAME]

for row, date in enumerate(REPORT_DAY_INDEX):
    excel_row = row + 2
    assert pd.Timestamp(plan_sheet.cell(excel_row, 1).value).normalize() == date
    assert pd.Timestamp(adjustment_sheet.cell(excel_row, 1).value).normalize() == date

    for column, value in enumerate(FINAL_POLICY["plan"][row], start=2):
        plan_sheet.cell(excel_row, column).value = round(float(value), 4)
    plan_sheet.cell(excel_row, 146).value = round(float(FINAL_POLICY["plan"][row].sum()), 4)
    plan_sheet.cell(excel_row, 147).value = round(float(FINAL_POLICY["plan_cost"][row]), 4)

    for column, value in enumerate(FINAL_POLICY["adjusted"][row], start=2):
        adjustment_sheet.cell(excel_row, column).value = round(float(value), 4)
    adjustment_sheet.cell(excel_row, 146).value = round(float(FINAL_POLICY["adjusted"][row].sum()), 4)
    adjustment_sheet.cell(excel_row, 147).value = round(float(FINAL_POLICY["adjustment_cost"][row]), 4)

battery_style_rows = [
    [copy.copy(battery_sheet.cell(row, column)._style) for column in range(1, 7)]
    for row in range(2, 8)
]
if battery_sheet.max_row > 1:
    battery_sheet.delete_rows(2, battery_sheet.max_row - 1)

for row, date in enumerate(REPORT_DAY_INDEX):
    charge_blocks = block_totals(FINAL_POLICY["charge"][row])
    discharge_blocks = block_totals(FINAL_POLICY["discharge"][row])
    first_row = 2 + row * 6
    for block, label in enumerate(FOUR_HOUR_LABELS):
        excel_row = first_row + block
        for column in range(1, 7):
            battery_sheet.cell(excel_row, column)._style = copy.copy(battery_style_rows[block][column - 1])
        if block == 0:
            battery_sheet.cell(excel_row, 1).value = date.to_pydatetime()
            battery_sheet.cell(excel_row, 5).value = "0:00"
            battery_sheet.cell(excel_row, 6).value = round(float(FINAL_POLICY["soc_open"][row]), 4)
        elif block == 1:
            battery_sheet.cell(excel_row, 5).value = "24:00"
            battery_sheet.cell(excel_row, 6).value = round(float(FINAL_POLICY["soc_close"][row]), 4)
        battery_sheet.cell(excel_row, 2).value = label
        battery_sheet.cell(excel_row, 3).value = round(float(charge_blocks[block]), 4)
        battery_sheet.cell(excel_row, 4).value = round(float(discharge_blocks[block]), 4)

emergency_style_rows = [
    [copy.copy(emergency_sheet.cell(row, column)._style) for column in range(1, 4)]
    for row in (2, 3, 4)
]
if emergency_sheet.max_row > 1:
    emergency_sheet.delete_rows(2, emergency_sheet.max_row - 1)

excel_row = 2
for row, date in enumerate(REPORT_DAY_INDEX):
    windows = merge_emergency_windows(FINAL_POLICY["emergency"][row])
    if not windows:
        windows = [("None", 0.0)]
    for window_index, (window, energy) in enumerate(windows):
        if len(windows) == 1:
            style_index = 0
        elif window_index == 0:
            style_index = 0
        elif window_index == len(windows) - 1:
            style_index = 2
        else:
            style_index = 1
        for column in range(1, 4):
            emergency_sheet.cell(excel_row, column)._style = copy.copy(emergency_style_rows[style_index][column - 1])
        if window_index == 0:
            emergency_sheet.cell(excel_row, 1).value = date.to_pydatetime()
        emergency_sheet.cell(excel_row, 2).value = window
        emergency_sheet.cell(excel_row, 3).value = round(float(energy), 4)
        excel_row += 1

result_book.save(RESULT_FILE)
result_book.close()
print("Written:", RESULT_FILE)

Written: C:\Users\lenovo\Downloads\CUMCM2026Problems\C题\q3_outputs\result3.xlsx


In [15]:
check_book = load_workbook(RESULT_FILE, read_only=True, data_only=True)
check_plan = check_book[PLAN_SHEET_NAME]
check_adjustment = check_book[ADJUSTMENT_SHEET_NAME]
check_battery = check_book[BATTERY_SHEET_NAME]
check_emergency = check_book[EMERGENCY_SHEET_NAME]

plan_matrix = np.array(
    [[cell.value for cell in row] for row in check_plan.iter_rows(min_row=2, max_row=335, min_col=2, max_col=145)],
    dtype=float,
)
adjustment_matrix = np.array(
    [[cell.value for cell in row] for row in check_adjustment.iter_rows(min_row=2, max_row=335, min_col=2, max_col=145)],
    dtype=float,
)
plan_totals = np.array(
    [row[0].value for row in check_plan.iter_rows(min_row=2, max_row=335, min_col=146, max_col=146)],
    dtype=float,
)
adjustment_totals = np.array(
    [row[0].value for row in check_adjustment.iter_rows(min_row=2, max_row=335, min_col=146, max_col=146)],
    dtype=float,
)

export_checks = pd.Series(
    {
        "plan matrix shape": plan_matrix.shape,
        "adjustment matrix shape": adjustment_matrix.shape,
        "missing plan cells": int(np.isnan(plan_matrix).sum()),
        "missing adjustment cells": int(np.isnan(adjustment_matrix).sum()),
        "maximum plan export difference (kWh)": np.abs(plan_matrix - FINAL_POLICY["plan"]).max(),
        "maximum adjustment export difference (kWh)": np.abs(adjustment_matrix - FINAL_POLICY["adjusted"]).max(),
        "maximum plan row-total error (kWh)": np.abs(plan_totals - plan_matrix.sum(axis=1)).max(),
        "maximum adjustment row-total error (kWh)": np.abs(adjustment_totals - adjustment_matrix.sum(axis=1)).max(),
        "battery data rows": check_battery.max_row - 1,
        "expected battery data rows": len(EVAL_DAYS) * 6,
        "emergency data rows": check_emergency.max_row - 1,
    },
    name="value",
)
display(export_checks.to_frame())

assert export_checks["missing plan cells"] == 0
assert export_checks["missing adjustment cells"] == 0
assert export_checks["maximum plan export difference (kWh)"] < 1e-3
assert export_checks["maximum adjustment export difference (kWh)"] < 1e-3
assert export_checks["maximum plan row-total error (kWh)"] < 1e-2
assert export_checks["maximum adjustment row-total error (kWh)"] < 1e-2
assert export_checks["battery data rows"] == len(EVAL_DAYS) * 6
check_book.close()
print("result3.xlsx was reopened and verified successfully.")

,value
plan matrix shape,"(334, 144)"
adjustment matrix shape,"(334, 144)"
missing plan cells,0
missing adjustment cells,0
maximum plan export difference (kWh),0.00005
maximum adjustment export difference (kWh),0.00005
maximum plan row-total error (kWh),0.001
maximum adjustment row-total error (kWh),0.001
battery data rows,2004
expected battery data rows,2004


result3.xlsx was reopened and verified successfully.


## 8. 结论：是否需要使用 6:00、12:00、18:00 的预报？

下面的单元格依据已经完成的全年真实结算自动给出结论。判断标准不是单独比较预测 MAE，而是比较两种策略在相同真实负荷、真实光伏和相同 0:00 计划下的总费用；这样同时计入日内调整费用和紧急购电节约。

In [16]:
static_cost = float(STATIC_POLICY["total_cost"].sum())
selected_cost = float(FINAL_POLICY["total_cost"].sum())
saving = static_cost - selected_cost
saving_rate = 100.0 * saving / static_cost
static_emergency = float(STATIC_POLICY["emergency"].sum())
selected_emergency = float(FINAL_POLICY["emergency"].sum())

if BEST_UPDATE_SUBSET:
    recommendation = f"Use intraday forecasts at {BEST_UPDATE_LABEL} for rolling adjustments."
    reason = "The reduction in emergency-purchase cost exceeds the added adjustment cost."
else:
    recommendation = "Keep the 00:00 plan and do not use routine intraday adjustments under the stated fees."
    reason = "The adjustment penalties exceed the emergency-purchase savings."

conclusion_text = f'''
### Executed conclusion

- **Recommendation:** {recommendation}
- **00:00-only total cost:** {static_cost:,.2f} yuan.
- **Selected-policy total cost:** {selected_cost:,.2f} yuan.
- **Saving from the selected updates:** {saving:,.2f} yuan ({saving_rate:.2f}%).
- **Emergency energy:** {static_emergency:,.2f} kWh without updates versus {selected_emergency:,.2f} kWh with the selected updates.
- **Economic explanation:** {reason}

The exported `result3.xlsx` uses the lowest-cost update set evaluated above.
'''
display(Markdown(conclusion_text))


### Executed conclusion

- **Recommendation:** Use intraday forecasts at 06:00 + 12:00 for rolling adjustments.
- **00:00-only total cost:** 14,935,980.76 yuan.
- **Selected-policy total cost:** 14,893,895.27 yuan.
- **Saving from the selected updates:** 42,085.49 yuan (0.28%).
- **Emergency energy:** 359,768.28 kWh without updates versus 279,286.68 kWh with the selected updates.
- **Economic explanation:** The reduction in emergency-purchase cost exceeds the added adjustment cost.

The exported `result3.xlsx` uses the lowest-cost update set evaluated above.
